### Building a RAG System with LangChain & ChromaDB

Introduction

Retrival-Augmented Generation (RAG) is a technique which combines the capabilites of LLMs with external knowledge. 
- LangChain : A framework for developing applications powered by Language models
- ChromaDB : An open source vector store for storing and retrieving embeddings 

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
# langchain imports
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_chroma import Chroma

# importing the ulities
import numpy as np
import matplotlib.pyplot as plt
from typing import List

/Users/namitkumar/programming/rag/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### RAG Architecture Overview

#### What is RAG?

Retrieval-Augmented Generation (RAG) is an AI architecture that enhances Large Language Models (LLMs) by providing relevant external knowledge during inference. Instead of relying solely on the model's training data, RAG retrieves relevant information from a knowledge base and uses it to generate accurate, context-aware responses.

---

#### RAG Workflow

text Documents     │     ▼ Document Loaders     │     ▼ Text Preprocessing     │     ▼ Text Chunking     │     ▼ Embedding Model     │     ▼ Vector Database     │     ▼ Retriever     │ User Query     │     ▼ Query Embedding     │     ▼ Similarity Search     │     ▼ Relevant Chunks     │     ▼ Prompt Construction     │     ▼ Large Language Model     │     ▼ Generated Response 

---

#### Components

##### 1. Document Loaders

Document loaders ingest data from various sources such as:

- PDF files
- Text documents
- Word documents
- CSV files
- Web pages
- Databases

The loader converts raw data into LangChain Document objects.

---

##### 2. Text Preprocessing

Raw text is cleaned before processing:

- Remove unnecessary whitespace
- Normalize formatting
- Remove noise and special characters
- Standardize document structure

This improves retrieval quality.

---

##### 3. Text Chunking

Large documents are divided into smaller chunks.

Common strategies:

- Character-based chunking
- Recursive chunking
- Token-based chunking
- Semantic chunking

Benefits:

- Better retrieval accuracy
- Reduced context window usage
- Improved relevance matching

---

##### 4. Embedding Generation

Each chunk is converted into a numerical vector representation using an embedding model.

Examples:

- sentence-transformers/all-MiniLM-L6-v2
- BAAI/bge-small-en-v1.5
- OpenAI Embeddings

Embeddings capture semantic meaning rather than exact keywords.

---

##### 5. Vector Database

Embeddings are stored in a vector database.

Popular options:

- Chroma
- FAISS
- Pinecone
- Weaviate
- Milvus

The vector database enables efficient similarity search.

---

##### 6. Retrieval

When a user submits a query:

1. The query is converted into an embedding.
2. Similarity search is performed.
3. The most relevant chunks are retrieved.

This process ensures that only relevant information is passed to the LLM.

---

##### 7. Prompt Augmentation

Retrieved chunks are combined with the user's query.

Example:

Context: [Retrieved Documents]  Question: [User Query]

This augmented prompt provides the LLM with relevant knowledge.

---

##### 8. Response Generation

The LLM uses:

- User question
- Retrieved context
- Prompt instructions

to generate an accurate and context-aware response.

---

#### Benefits of RAG

- Reduces hallucinations
- Uses up-to-date information
- Enables domain-specific knowledge
- Improves response accuracy
- Works with private enterprise data
- Reduces dependency on model retraining

---

#### Technology Stack

##### Data Ingestion

- LangChain Document Loaders
- PyPDFLoader
- TextLoader

##### Text Processing

- RecursiveCharacterTextSplitter
- TokenTextSplitter

##### Embeddings

- HuggingFace Embeddings
- Sentence Transformers

##### Vector Store

- ChromaDB
- FAISS

##### LLM

- Groq
- Ollama
- OpenAI

##### Framework

- LangChain
- LangGraph

---

#### Future Enhancements

- Hybrid Search
- Multi-Query Retrieval
- Context Compression
- Reranking
- Multi-Agent RAG
- Multimodal RAG
- Conversational Memory
- Knowledge Graph Integration

---

#### Conclusion

RAG combines retrieval systems with Large Language Models to create intelligent applications capable of generating accurate, context-aware responses from external knowledge sources. It forms the foundation of modern AI assistants, document chatbots, enterprise search systems, and agentic AI workflows.

### 1. Sample Data

In [3]:
# create the sample data
sample_data = [
    """
    Python is a high-level, interpreted programming language known for its
    simplicity, readability, and versatility. It supports multiple programming
    paradigms including procedural, object-oriented, and functional programming.

    Python is widely used in web development, artificial intelligence,
    machine learning, data science, automation, cybersecurity, and scientific
    computing. Popular frameworks include Django, Flask, FastAPI, and Streamlit.

    The Python ecosystem contains thousands of open-source libraries that
    accelerate development and make it one of the most popular programming
    languages in the world.
    """,

    """
    Retrieval-Augmented Generation (RAG) is an architecture that combines
    information retrieval with large language models. Instead of relying solely
    on the model's training data, RAG systems retrieve relevant information
    from an external knowledge base.

    The retrieved documents are provided to the language model as additional
    context before generating a response. This improves factual accuracy,
    reduces hallucinations, and enables the model to answer questions about
    proprietary or recently updated information.

    Modern RAG systems often use vector databases, embeddings, reranking,
    hybrid search, and contextual compression to improve retrieval quality.
    """,

    """
    LangChain is an open-source framework designed to simplify the development
    of applications powered by large language models. It provides abstractions
    for prompts, chains, agents, memory systems, document loaders, and
    retrieval pipelines.

    Developers can integrate multiple AI models, external APIs, databases,
    and tools into a unified workflow. LangChain is widely used for building
    chatbots, question-answering systems, AI assistants, and RAG applications.

    The framework works with providers such as OpenAI, Anthropic, Groq,
    Ollama, Hugging Face, and many others.
    """,

    """
    Vector databases are specialized storage systems designed to manage and
    search high-dimensional vector embeddings efficiently. They play a crucial
    role in modern AI applications, particularly Retrieval-Augmented Generation.

    When documents are converted into embeddings, they are stored inside a
    vector database. User queries are also transformed into embeddings and
    compared using similarity search algorithms.

    Popular vector databases include ChromaDB, Pinecone, Weaviate, Milvus,
    Qdrant, and FAISS. These systems enable semantic search rather than
    traditional keyword matching.
    """,

    """
    Machine Learning is a branch of Artificial Intelligence focused on building
    systems that learn patterns from data. Instead of explicitly programming
    every rule, developers train models using datasets and optimization
    algorithms.

    Common categories include supervised learning, unsupervised learning,
    reinforcement learning, and self-supervised learning. Machine learning
    powers recommendation systems, fraud detection, computer vision, speech
    recognition, and predictive analytics.

    The rapid growth of machine learning has significantly contributed to the
    development of modern AI systems and large language models.
    """
]

In [4]:
# save the sample data to a text file
import tempfile
temp_dir = tempfile.mkdtemp()

for i, doc in enumerate(sample_data):
    with open(os.path.join(temp_dir, f"doc_{i}.txt"), "w") as f:
        f.write(doc)

print(f"Sample data saved to: {temp_dir}")

Sample data saved to: /var/folders/zb/z5p3pbrx5xl_bdxy0pkbj91m0000gn/T/tmp6jy3eeu8


In [5]:
temp_dir

'/var/folders/zb/z5p3pbrx5xl_bdxy0pkbj91m0000gn/T/tmp6jy3eeu8'

### 2. Document Loading

In [6]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

# load the documents using the TextLoader
loader = DirectoryLoader(
    temp_dir, 
    glob="*.txt",
    show_progress=True,
    loader_kwargs={"encoding": "utf-8"},
    loader_cls= TextLoader
)
documents = loader.load()

print(f"Number of documents loaded: {len(documents)}")
print(f"First document content:\n{documents[0].page_content[:500]}...")


100%|██████████| 5/5 [00:00<00:00, 3958.38it/s]

Number of documents loaded: 5
First document content:

    Vector databases are specialized storage systems designed to manage and
    search high-dimensional vector embeddings efficiently. They play a crucial
    role in modern AI applications, particularly Retrieval-Augmented Generation.

    When documents are converted into embeddings, they are stored inside a
    vector database. User queries are also transformed into embeddings and
    compared using similarity search algorithms.

    Popular vector databases include ChromaDB, Pinecone, Weavi...


In [7]:
documents

[Document(metadata={'source': '/var/folders/zb/z5p3pbrx5xl_bdxy0pkbj91m0000gn/T/tmp6jy3eeu8/doc_3.txt'}, page_content='\n    Vector databases are specialized storage systems designed to manage and\n    search high-dimensional vector embeddings efficiently. They play a crucial\n    role in modern AI applications, particularly Retrieval-Augmented Generation.\n\n    When documents are converted into embeddings, they are stored inside a\n    vector database. User queries are also transformed into embeddings and\n    compared using similarity search algorithms.\n\n    Popular vector databases include ChromaDB, Pinecone, Weaviate, Milvus,\n    Qdrant, and FAISS. These systems enable semantic search rather than\n    traditional keyword matching.\n    '),
 Document(metadata={'source': '/var/folders/zb/z5p3pbrx5xl_bdxy0pkbj91m0000gn/T/tmp6jy3eeu8/doc_2.txt'}, page_content='\n    LangChain is an open-source framework designed to simplify the development\n    of applications powered by large lang

### 3. Document Splitter

In [8]:
# intialize the text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500, 
    chunk_overlap=50, 
    length_function=len,
    separators=[" "]
)

chunks = text_splitter.split_documents(documents)

print(f"Number of chunks created: {len(chunks)}")
print(f"First chunk content:\n{chunks[0].page_content[:500]}...")

Number of chunks created: 10
First chunk content:
Vector databases are specialized storage systems designed to manage and
    search high-dimensional vector embeddings efficiently. They play a crucial
    role in modern AI applications, particularly Retrieval-Augmented Generation.

    When documents are converted into embeddings, they are stored inside a
    vector database. User queries are also transformed into embeddings and
    compared using similarity search algorithms.

    Popular vector databases include ChromaDB, Pinecone,...


In [9]:
chunks

[Document(metadata={'source': '/var/folders/zb/z5p3pbrx5xl_bdxy0pkbj91m0000gn/T/tmp6jy3eeu8/doc_3.txt'}, page_content='Vector databases are specialized storage systems designed to manage and\n    search high-dimensional vector embeddings efficiently. They play a crucial\n    role in modern AI applications, particularly Retrieval-Augmented Generation.\n\n    When documents are converted into embeddings, they are stored inside a\n    vector database. User queries are also transformed into embeddings and\n    compared using similarity search algorithms.\n\n    Popular vector databases include ChromaDB, Pinecone,'),
 Document(metadata={'source': '/var/folders/zb/z5p3pbrx5xl_bdxy0pkbj91m0000gn/T/tmp6jy3eeu8/doc_3.txt'}, page_content='vector databases include ChromaDB, Pinecone, Weaviate, Milvus,\n    Qdrant, and FAISS. These systems enable semantic search rather than\n    traditional keyword matching.'),
 Document(metadata={'source': '/var/folders/zb/z5p3pbrx5xl_bdxy0pkbj91m0000gn/T/tmp6jy3

### 4. Embedding Model

In [10]:
# Initialize the HuggingFaceEmbeddings with a specific model
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
print(embedding_model)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9959.27it/s]


model_name='sentence-transformers/all-MiniLM-L6-v2' cache_folder=None model_kwargs={} encode_kwargs={} query_encode_kwargs={} multi_process=False show_progress=False


#### Initialize the ChromaDB Vector Store to store the chunks in vector representation

In [11]:
# Initialize the Chroma vector database
persist_directory="./chroma_db"
vectorestore = Chroma.from_documents(
    collection_name="rag_documents",
    documents=chunks,
    embedding = embedding_model,
    persist_directory=persist_directory
)

print("Vector store created and documents indexed successfully!")
print(f"Collection name: {vectorestore._collection_name}")
print(f"Number of documents in the vector store: {vectorestore._collection.count()}")
print(f"Persist directory: {persist_directory}")

Vector store created and documents indexed successfully!
Collection name: rag_documents
Number of documents in the vector store: 60
Persist directory: ./chroma_db


#### Text Similarity Search

In [12]:
query = "What is Python used for?"

similar_docs = vectorestore.similarity_search(query, k=3)
similar_docs

[Document(id='26435e8f-1c4e-4ed6-9e9d-8f32bf048563', metadata={'source': '/var/folders/zb/z5p3pbrx5xl_bdxy0pkbj91m0000gn/T/tmp0ll_i6mq/doc_0.txt'}, page_content='Python is a high-level, interpreted programming language known for its\n    simplicity, readability, and versatility. It supports multiple programming\n    paradigms including procedural, object-oriented, and functional programming.\n\n    Python is widely used in web development, artificial intelligence,\n    machine learning, data science, automation, cybersecurity, and scientific\n    computing. Popular frameworks include Django, Flask, FastAPI, and Streamlit.\n\n    The Python ecosystem'),
 Document(id='1930b8ba-a1a7-44e9-b1d3-de58020eef01', metadata={'source': '/var/folders/zb/z5p3pbrx5xl_bdxy0pkbj91m0000gn/T/tmp0ll_i6mq/doc_0.txt'}, page_content='Python is a high-level, interpreted programming language known for its\n    simplicity, readability, and versatility. It supports multiple programming\n    paradigms including p

#### Advanced Similarity with Score

In [14]:
vectorestore.similarity_search_with_score(query, k=3)

[(Document(id='26435e8f-1c4e-4ed6-9e9d-8f32bf048563', metadata={'source': '/var/folders/zb/z5p3pbrx5xl_bdxy0pkbj91m0000gn/T/tmp0ll_i6mq/doc_0.txt'}, page_content='Python is a high-level, interpreted programming language known for its\n    simplicity, readability, and versatility. It supports multiple programming\n    paradigms including procedural, object-oriented, and functional programming.\n\n    Python is widely used in web development, artificial intelligence,\n    machine learning, data science, automation, cybersecurity, and scientific\n    computing. Popular frameworks include Django, Flask, FastAPI, and Streamlit.\n\n    The Python ecosystem'),
  0.40170004963874817),
 (Document(id='1930b8ba-a1a7-44e9-b1d3-de58020eef01', metadata={'source': '/var/folders/zb/z5p3pbrx5xl_bdxy0pkbj91m0000gn/T/tmp0ll_i6mq/doc_0.txt'}, page_content='Python is a high-level, interpreted programming language known for its\n    simplicity, readability, and versatility. It supports multiple programming\

In [15]:
from langchain_groq import ChatGroq

# Initialize the ChatGroq client
llm = ChatGroq(model="llama-3.3-70b-versatile")

In [16]:
# test llm
test_lm = llm.invoke("What is the capital of France?")
print(test_lm)

content='The capital of France is Paris.' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 42, 'total_tokens': 50, 'completion_time': 0.010934147, 'completion_tokens_details': None, 'prompt_time': 0.003891134, 'prompt_tokens_details': None, 'queue_time': 0.161083566, 'total_time': 0.014825281}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019e838a-ea96-7462-80dd-d7b4016aae8a-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 42, 'output_tokens': 8, 'total_tokens': 50}


### Modern RAG Chain

In [18]:
from langchain.chains.retrieval import RetrievalQA

ModuleNotFoundError: No module named 'langchain.chains'